# Airline Operations — Power BI Star Schema

- `FACT_FLIGHT_OPERATIONS`
- `DIM_DATE`
- `DIM_CARRIER`
- `DIM_ORIGIN_AIRPORT`
- `DIM_DEST_AIRPORT`
- `DIM_WEATHER`

The fact-table grain is **one row = one flight operation**.


In [1]:
import pandas as pd
from pathlib import Path


## 1. Load the merged source file



In [ ]:
SOURCE_FILE = r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\processed\flights_2026_q1.csv"

data = pd.read_csv(SOURCE_FILE, low_memory=False)

print("Rows:", len(data))
print("Columns:", len(data.columns))
data.head()


Rows: 1847242
Columns: 78


,year,quarter,month,day_of_month,day_of_week,fl_date,mkt_unique_carrier,branded_code_share,mkt_carrier_airline_id,mkt_carrier,...,dep_minute,scheduled_departure,departure_hour,temperature,precipitation,wind_speed,weather_code,weather_matched,weather_category,bad_weather
0,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,0,2026-01-01 07:00:00,7,0.9,0.0,18.3,3,True,Cloudy,0
1,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,30,2026-01-01 21:30:00,21,17.3,0.2,4.5,51,True,Rain,1
2,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,45,2026-01-01 22:45:00,22,18.2,0.0,7.9,0,True,Clear,0
3,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,38,2026-01-01 23:38:00,23,12.5,0.0,4.8,2,True,Cloudy,0
4,2026,1,1,1,4,2026-01-01,AA,AA,19805,AA,...,57,2026-01-01 07:57:00,7,-2.4,0.1,16.7,71,True,Snow,1


## 2. Basic date and type preparation

Keep this simple. Power BI will also apply data types when the CSVs are loaded.


In [3]:
data["fl_date"] = pd.to_datetime(data["fl_date"], errors="coerce")

# Make sure the weather matching hour is numeric
data["departure_hour"] = pd.to_numeric(data["departure_hour"], errors="coerce")

# Keep boolean columns consistent
bool_cols = [
    "missing_operational_duration",
    "extreme_dep_delay",
    "extreme_arr_delay",
    "weather_matched"
]

for col in bool_cols:
    if col in data.columns:
        data[col] = data[col].astype("boolean")


## 3. Create DIM_DATE

One row per calendar date.


In [4]:
DIM_DATE = (
    data[[
        "fl_date",
        "year",
        "quarter",
        "month",
        "day_of_month",
        "day_of_week"
    ]]
    .drop_duplicates("fl_date")
    .sort_values("fl_date")
    .reset_index(drop=True)
)

DIM_DATE = DIM_DATE.rename(columns={
    "fl_date": "Date"
})

print("DIM_DATE rows:", len(DIM_DATE))
DIM_DATE.head()


DIM_DATE rows: 90


,Date,year,quarter,month,day_of_month,day_of_week
0,2026-01-01,2026,1,1,1,4
1,2026-01-02,2026,1,1,2,5
2,2026-01-03,2026,1,1,3,6
3,2026-01-04,2026,1,1,4,7
4,2026-01-05,2026,1,1,5,1


## 4. Create DIM_CARRIER

One row per carrier ID.


In [5]:
DIM_CARRIER = (
    data[[
        "mkt_carrier_airline_id",
        "mkt_unique_carrier",
        "branded_code_share",
        "mkt_carrier"
    ]]
    .drop_duplicates("mkt_carrier_airline_id")
    .reset_index(drop=True)
)

DIM_CARRIER = DIM_CARRIER.rename(columns={
    "mkt_carrier_airline_id": "Carrier_ID",
    "mkt_unique_carrier": "Carrier_Name",
    "branded_code_share": "Branded_Code_Share",
    "mkt_carrier": "Carrier_Code"
})

print("DIM_CARRIER rows:", len(DIM_CARRIER))
DIM_CARRIER.head()


DIM_CARRIER rows: 9


,Carrier_ID,Carrier_Name,Branded_Code_Share,Carrier_Code
0,19805,AA,AA,AA
1,19930,AS,AS,AS
2,20409,B6,B6,B6
3,19790,DL,DL,DL
4,20436,F9,F9,F9


## 5. Create DIM_ORIGIN_AIRPORT

One row per origin airport.


In [6]:
DIM_ORIGIN_AIRPORT = (
    data[[
        "origin",
        "origin_airport_name",
        "origin_city_name",
        "origin_state_abr",
        "origin_state_nm",
        "origin_city",
        "origin_country",
        "origin_latitude",
        "origin_longitude",
        "origin_timezone"
    ]]
    .drop_duplicates("origin")
    .reset_index(drop=True)
)

DIM_ORIGIN_AIRPORT = DIM_ORIGIN_AIRPORT.rename(columns={
    "origin": "Airport_Code",
    "origin_airport_name": "Airport_Name",
    "origin_city_name": "City_Name",
    "origin_state_abr": "State_Code",
    "origin_state_nm": "State_Name",
    "origin_city": "City",
    "origin_country": "Country",
    "origin_latitude": "Latitude",
    "origin_longitude": "Longitude",
    "origin_timezone": "Timezone"
})

print("DIM_ORIGIN_AIRPORT rows:", len(DIM_ORIGIN_AIRPORT))
DIM_ORIGIN_AIRPORT.head()


DIM_ORIGIN_AIRPORT rows: 362


,Airport_Code,Airport_Name,City_Name,State_Code,State_Name,City,Country,Latitude,Longitude,Timezone
0,JFK,John F Kennedy International Airport,"New York, NY",NY,New York,New York,United States,40.639801,-73.778900,-5
1,LAX,Los Angeles International Airport,"Los Angeles, CA",CA,California,Los Angeles,United States,33.942501,-118.407997,-8
2,MIA,Miami International Airport,"Miami, FL",FL,Florida,Miami,United States,25.793200,-80.290604,-5
3,DEN,Denver International Airport,"Denver, CO",CO,Colorado,Denver,United States,39.861698,-104.672997,-7
4,BOS,General Edward Lawrence Logan International Ai...,"Boston, MA",MA,Massachusetts,Boston,United States,42.364300,-71.005203,-5


## 6. Create DIM_DEST_AIRPORT

One row per destination airport.


In [7]:
DIM_DEST_AIRPORT = (
    data[[
        "dest",
        "dest_airport_name",
        "dest_city_name",
        "dest_state_abr",
        "dest_state_nm",
        "dest_city",
        "dest_country",
        "dest_latitude",
        "dest_longitude",
        "dest_timezone"
    ]]
    .drop_duplicates("dest")
    .reset_index(drop=True)
)

DIM_DEST_AIRPORT = DIM_DEST_AIRPORT.rename(columns={
    "dest": "Airport_Code",
    "dest_airport_name": "Airport_Name",
    "dest_city_name": "City_Name",
    "dest_state_abr": "State_Code",
    "dest_state_nm": "State_Name",
    "dest_city": "City",
    "dest_country": "Country",
    "dest_latitude": "Latitude",
    "dest_longitude": "Longitude",
    "dest_timezone": "Timezone"
})

print("DIM_DEST_AIRPORT rows:", len(DIM_DEST_AIRPORT))
DIM_DEST_AIRPORT.head()


DIM_DEST_AIRPORT rows: 362


,Airport_Code,Airport_Name,City_Name,State_Code,State_Name,City,Country,Latitude,Longitude,Timezone
0,LAX,Los Angeles International Airport,"Los Angeles, CA",CA,California,Los Angeles,United States,33.942501,-118.407997,-8
1,JFK,John F Kennedy International Airport,"New York, NY",NY,New York,New York,United States,40.639801,-73.778900,-5
2,MSY,Louis Armstrong New Orleans International Airport,"New Orleans, LA",LA,Louisiana,New Orleans,United States,29.993401,-90.258003,-6
3,MIA,Miami International Airport,"Miami, FL",FL,Florida,Miami,United States,25.793200,-80.290604,-5
4,CLT,Charlotte Douglas International Airport,"Charlotte, NC",NC,North Carolina,Charlotte,United States,35.214001,-80.943100,-5


## 7. Create DIM_WEATHER

Weather grain:

**Origin airport + flight date + departure hour**

This prevents the same weather observation from being repeated for every flight.


In [8]:
weather_cols = [
    "origin",
    "fl_date",
    "departure_hour",
    "temperature",
    "precipitation",
    "wind_speed",
    "weather_code",
    "weather_matched",
    "weather_category",
    "bad_weather"
]

DIM_WEATHER = (
    data[weather_cols]
    .drop_duplicates(["origin", "fl_date", "departure_hour"])
    .reset_index(drop=True)
)

DIM_WEATHER.insert(0, "Weather_Key", range(1, len(DIM_WEATHER) + 1))

DIM_WEATHER = DIM_WEATHER.rename(columns={
    "origin": "Airport_Code",
    "fl_date": "Weather_Date",
    "departure_hour": "Weather_Hour",
    "temperature": "Temperature",
    "precipitation": "Precipitation",
    "wind_speed": "Wind_Speed",
    "weather_code": "Weather_Code",
    "weather_matched": "Weather_Matched",
    "weather_category": "Weather_Category",
    "bad_weather": "Bad_Weather"
})

print("DIM_WEATHER rows:", len(DIM_WEATHER))
DIM_WEATHER.head()


DIM_WEATHER rows: 274031


,Weather_Key,Airport_Code,Weather_Date,Weather_Hour,Temperature,Precipitation,Wind_Speed,Weather_Code,Weather_Matched,Weather_Category,Bad_Weather
0,1,JFK,2026-01-01,7,0.9,0.0,18.3,3,True,Cloudy,0
1,2,LAX,2026-01-01,21,17.3,0.2,4.5,51,True,Rain,1
2,3,MIA,2026-01-01,22,18.2,0.0,7.9,0,True,Clear,0
3,4,DEN,2026-01-01,23,12.5,0.0,4.8,2,True,Cloudy,0
4,5,BOS,2026-01-01,7,-2.4,0.1,16.7,71,True,Snow,1


## 8. Add Weather_Key to the fact table

The fact table keeps only the `Weather_Key`, not the repeated weather attributes.


In [9]:
weather_lookup = DIM_WEATHER.rename(columns={
    "Airport_Code": "origin",
    "Weather_Date": "fl_date",
    "Weather_Hour": "departure_hour"
})[[
    "origin",
    "fl_date",
    "departure_hour",
    "Weather_Key"
]]

data_with_weather = data.merge(
    weather_lookup,
    on=["origin", "fl_date", "departure_hour"],
    how="left",
    validate="many_to_one"
)

print("Source rows:", len(data))
print("Rows after weather merge:", len(data_with_weather))

if len(data) != len(data_with_weather):
    raise ValueError("Weather merge changed the fact row count. Check the weather grain.")


Source rows: 1847242
Rows after weather merge: 1847242


## 9. Create FACT_FLIGHT_OPERATIONS

One row remains one flight operation.


In [10]:
FACT_COLUMNS = [
    "fl_date",
    "mkt_carrier_airline_id",
    "mkt_carrier_fl_num",
    "origin",
    "dest",
    "Weather_Key",

    "crs_dep_time",
    "dep_time",
    "dep_delay",
    "dep_delay_new",
    "dep_del15",
    "taxi_out",
    "taxi_in",

    "crs_arr_time",
    "arr_time",
    "arr_delay",
    "arr_delay_new",
    "arr_del15",

    "cancelled",
    "cancellation_code",
    "diverted",
    "crs_elapsed_time",
    "actual_elapsed_time",
    "air_time",
    "distance",

    "carrier_delay",
    "weather_delay",
    "nas_delay",
    "security_delay",
    "late_aircraft_delay",

    "flight_status",
    "dep_time_at_cancellation",

    "missing_operational_duration",
    "extreme_dep_delay",
    "extreme_arr_delay",
    "is_cancelled",
    "is_diverted",
    "is_irregular_operation",
    "is_departure_delayed",
    "is_arrival_delayed",
    "severe_departure_delay",
    "severe_arrival_delay"
]

FACT_FLIGHT_OPERATIONS = data_with_weather[FACT_COLUMNS].copy()

print("FACT_FLIGHT_OPERATIONS rows:", len(FACT_FLIGHT_OPERATIONS))
print("FACT_FLIGHT_OPERATIONS columns:", len(FACT_FLIGHT_OPERATIONS.columns))
FACT_FLIGHT_OPERATIONS.head()


FACT_FLIGHT_OPERATIONS rows: 1847242
FACT_FLIGHT_OPERATIONS columns: 42


,fl_date,mkt_carrier_airline_id,mkt_carrier_fl_num,origin,dest,Weather_Key,crs_dep_time,dep_time,dep_delay,dep_delay_new,...,missing_operational_duration,extreme_dep_delay,extreme_arr_delay,is_cancelled,is_diverted,is_irregular_operation,is_departure_delayed,is_arrival_delayed,severe_departure_delay,severe_arrival_delay
0,2026-01-01,19805,1,JFK,LAX,1,700,657.0,-3.0,0.0,...,False,False,False,0,0,0,0,1,0,0
1,2026-01-01,19805,10,LAX,JFK,2,2130,2128.0,-2.0,0.0,...,False,False,False,0,0,0,0,0,0,0
2,2026-01-01,19805,1002,MIA,MSY,3,2245,2252.0,7.0,7.0,...,False,False,False,0,0,0,0,0,0,0
3,2026-01-01,19805,1003,DEN,MIA,4,2338,2338.0,0.0,0.0,...,False,False,False,0,0,0,0,0,0,0
4,2026-01-01,19805,1004,BOS,CLT,5,757,756.0,-1.0,0.0,...,False,False,False,0,0,0,0,0,0,0


## 10. Validate the star-schema keys

These checks should pass before loading the tables into Power BI.


In [11]:
print("FACT rows:", len(FACT_FLIGHT_OPERATIONS))
print("Source rows:", len(data))
print("Row count matches:", len(FACT_FLIGHT_OPERATIONS) == len(data))

print("\nDIM_DATE duplicate dates:", DIM_DATE["Date"].duplicated().sum())
print("DIM_CARRIER duplicate IDs:", DIM_CARRIER["Carrier_ID"].duplicated().sum())
print("DIM_ORIGIN_AIRPORT duplicate codes:", DIM_ORIGIN_AIRPORT["Airport_Code"].duplicated().sum())
print("DIM_DEST_AIRPORT duplicate codes:", DIM_DEST_AIRPORT["Airport_Code"].duplicated().sum())
print("DIM_WEATHER duplicate keys:", DIM_WEATHER["Weather_Key"].duplicated().sum())

print("\nUnmatched weather keys:",
      FACT_FLIGHT_OPERATIONS["Weather_Key"].isna().sum())


FACT rows: 1847242
Source rows: 1847242
Row count matches: True

DIM_DATE duplicate dates: 0
DIM_CARRIER duplicate IDs: 0
DIM_ORIGIN_AIRPORT duplicate codes: 0
DIM_DEST_AIRPORT duplicate codes: 0
DIM_WEATHER duplicate keys: 0

Unmatched weather keys: 0


## 11. Save the six tables

The files are ready to import into Power BI.


In [14]:
OUTPUT_DIR = Path(r"D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema")
OUTPUT_DIR.mkdir(exist_ok=True)

tables = {
    "FACT_FLIGHT_OPERATIONS": FACT_FLIGHT_OPERATIONS,
    "DIM_DATE": DIM_DATE,
    "DIM_CARRIER": DIM_CARRIER,
    "DIM_ORIGIN_AIRPORT": DIM_ORIGIN_AIRPORT,
    "DIM_DEST_AIRPORT": DIM_DEST_AIRPORT,
    "DIM_WEATHER": DIM_WEATHER
}

for name, table in tables.items():
    file_path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(file_path, index=False)
    print(f"Saved: {file_path}")


Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\FACT_FLIGHT_OPERATIONS.csv
Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\DIM_DATE.csv
Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\DIM_CARRIER.csv
Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\DIM_ORIGIN_AIRPORT.csv
Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\DIM_DEST_AIRPORT.csv
Saved: D:\Data Analyst\EXCEL\airline-operations-intelligence\Airline-Operations-Intelligence\data\airline_star_schema\DIM_WEATHER.csv


# Power BI relationships

Create these relationships in Model View:

1. `DIM_DATE[Date]` → `FACT_FLIGHT_OPERATIONS[fl_date]`
2. `DIM_CARRIER[Carrier_ID]` → `FACT_FLIGHT_OPERATIONS[mkt_carrier_airline_id]`
3. `DIM_ORIGIN_AIRPORT[Airport_Code]` → `FACT_FLIGHT_OPERATIONS[origin]`
4. `DIM_DEST_AIRPORT[Airport_Code]` → `FACT_FLIGHT_OPERATIONS[dest]`
5. `DIM_WEATHER[Weather_Key]` → `FACT_FLIGHT_OPERATIONS[Weather_Key]`

For all five:

- Cardinality: **One-to-many (1:*)**
- Cross-filter direction: **Single**
- Active relationship: **Yes**

The result is a simple star schema with the fact table in the center.


In [15]:
print("STAR SCHEMA READY")
for name, table in tables.items():
    print(f"{name}: {len(table):,} rows × {len(table.columns)} columns")


STAR SCHEMA READY
FACT_FLIGHT_OPERATIONS: 1,847,242 rows × 42 columns
DIM_DATE: 90 rows × 6 columns
DIM_CARRIER: 9 rows × 4 columns
DIM_ORIGIN_AIRPORT: 362 rows × 10 columns
DIM_DEST_AIRPORT: 362 rows × 10 columns
DIM_WEATHER: 274,031 rows × 11 columns
